# Notebook 3 · Basic LangChain

### Tools, tool choice, and the schema the model actually reads

Notebook 2 gave an agent one tool. Here we go deep on tools, because tools are where an agent stops guessing and starts knowing. You will see what a tool looks like to the model, watch an agent work through several tools, and learn the single habit that decides whether tool choice works: the docstring.

**Standalone setup.** Every notebook in this series stands on its own. Run the cell below first.

```bash
pip install langchain langgraph langchain-aws
```

In [1]:
from typing import Any, List
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.outputs import ChatResult, ChatGeneration
from langchain.agents import create_agent
from langchain.tools import tool


class ScriptedChatModel(BaseChatModel):
    '''Deterministic, credential-free stand-in. The framework code is real;
    only the model replies are scripted. Swap for ChatBedrockConverse in production.'''
    responses: List[Any]
    idx: int = 0

    @property
    def _llm_type(self) -> str:
        return "scripted"

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        reply = self.responses[min(self.idx, len(self.responses) - 1)]
        object.__setattr__(self, "idx", self.idx + 1)
        return ChatResult(generations=[ChatGeneration(message=reply)])

    def bind_tools(self, tools, **kwargs):
        return self


def show_trace(result):
    for m in result["messages"]:
        label = type(m).__name__
        if getattr(m, "tool_calls", None):
            c = m.tool_calls[0]
            print(f"{label:14} -> calls {c['name']}({c['args']})")
        else:
            print(f"{label:14} -> {m.content}")


print("setup ready")

setup ready


---
## 1. A tool is a function the model can read

The `@tool` decorator turns a Python function into something the model can see and request. Three parts matter, and the model only ever sees the first two:

| Part | Comes from | The model uses it to |
|------|-----------|----------------------|
| name | the function name | refer to the tool |
| description | the docstring | decide when to call it |
| args schema | the type hints | fill in the arguments correctly |

The function body is invisible to the model. It runs on your machine after the model asks.

```mermaid
flowchart LR
    F["@tool def lookup_pnr(pnr: str)"] --> S["schema the model sees: name + description + args"]
    S --> D[model decides: call it or not]
    D --> B[your body runs, returns a result]
```

> **What runs next** define a tool and print the exact schema the model reads. This is not documentation, it is the live contract.
> **Python construct** a decorated function, plus reading generated attributes off it.
> **LLM concept** tool calling works by schema. The model matches the user's need against these descriptions.

In [2]:
@tool
def lookup_pnr(pnr: str) -> str:
    '''Return the current booking status for a passenger name record (PNR).'''
    records = {"JX48Q2": "BLR-DEL cancelled, Rao, Gold tier"}
    return records.get(pnr, "PNR not found")


print("name        :", lookup_pnr.name)
print("description :", lookup_pnr.description)
print("args schema :", lookup_pnr.args)

name        : lookup_pnr
description : Return the current booking status for a passenger name record (PNR).
args schema : {'pnr': {'title': 'Pnr', 'type': 'string'}}


> **What just happened** the model does not see your dictionary or your return statement. It sees a name, one sentence, and an argument called `pnr` of type string. That sentence is doing the heavy lifting: it is how the model decides this is the right tool for "is my booking cancelled?".

> **Gotcha** rename the tool to `f1` and delete the docstring, and a real model will call it at the wrong times or not at all. The schema is the only thing standing between the user's words and the right function. Write docstrings like the model's job depends on them, because it does.

---
## 2. Several tools, worked in sequence

Real agents hold a small set of tools and pick among them. TravelMind needs to look up the booking, then search for alternatives. With a scripted model we fix the sequence, but the schema the model would use to choose is exactly the one you just printed.

```mermaid
flowchart TD
    U[cancelled, PNR JX48Q2, options?] --> M1[model: lookup_pnr]
    M1 --> T1[record: BLR-DEL cancelled]
    T1 --> M2[model: search_flights]
    M2 --> T2[AI-506 09:40, AI-812 14:15]
    T2 --> M3[model: grounded answer]
```

> **What runs next** add a second tool, then run an agent that calls both before answering.
> **Python construct** a list of tools passed to `create_agent`.
> **Language concept** discourse structure: the passenger's one sentence implies an ordered plan (find the booking, then find replacements). The agent recovers that order.

In [3]:
@tool
def search_flights(origin: str, dest: str, date: str) -> str:
    '''Find alternate flights between two airport codes on a given date.'''
    return "AI-506 09:40, AI-812 14:15"


model = ScriptedChatModel(responses=[
    AIMessage(content="", tool_calls=[{"name": "lookup_pnr", "args": {"pnr": "JX48Q2"}, "id": "a", "type": "tool_call"}]),
    AIMessage(content="", tool_calls=[{"name": "search_flights", "args": {"origin": "BLR", "dest": "DEL", "date": "today"}, "id": "b", "type": "tool_call"}]),
    AIMessage(content="JX48Q2 (BLR-DEL) is cancelled. Alternatives today: AI-506 09:40 or AI-812 14:15. Gold tier means no rebooking fee."),
])

agent = create_agent(model, tools=[lookup_pnr, search_flights], system_prompt="You are TravelMind.")
result = agent.invoke({"messages": [{"role": "user", "content": "My flight JX48Q2 is cancelled, what are my options?"}]})
show_trace(result)

HumanMessage   -> My flight JX48Q2 is cancelled, what are my options?
AIMessage      -> calls lookup_pnr({'pnr': 'JX48Q2'})
ToolMessage    -> BLR-DEL cancelled, Rao, Gold tier
AIMessage      -> calls search_flights({'origin': 'BLR', 'dest': 'DEL', 'date': 'today'})
ToolMessage    -> AI-506 09:40, AI-812 14:15
AIMessage      -> JX48Q2 (BLR-DEL) is cancelled. Alternatives today: AI-506 09:40 or AI-812 14:15. Gold tier means no rebooking fee.


> **What just happened** six messages, two tool round-trips, one grounded answer. The agent did not answer from memory. It gathered facts, then wrote. Every claim in the final message traces back to a `ToolMessage` above it. That traceability is the whole point of tools.

---
## 3. Docstrings decide tool choice

This is the lesson that separates working agents from flaky ones. When two tools could plausibly match, the model breaks the tie on the description. Watch what a real model has to work with when a docstring is lazy.

> **What runs next** define one clear tool and one vague tool, then print what the model sees for each.
> **Python construct** comparing two objects' generated schemas.
> **LLM concept** tool selection is a matching problem over descriptions. Ambiguous descriptions produce ambiguous behaviour.

In [4]:
@tool
def cancel_booking(pnr: str) -> str:
    '''Permanently cancel the booking for this PNR and start a refund. Irreversible.'''
    return f"{pnr} cancelled, refund started"


@tool
def handle(pnr: str) -> str:
    '''Handle the booking.'''
    return "done"


for t in (cancel_booking, handle):
    print(f"{t.name:16} | {t.description}")

cancel_booking   | Permanently cancel the booking for this PNR and start a refund. Irreversible.
handle           | Handle the booking.


> **What just happened** faced with "can you sort out my booking?", a real model reads both descriptions. `cancel_booking` is unambiguous: it says what it does and warns that it is irreversible. `handle` says nothing. A model might fire `handle` and silently do the wrong thing, or worse, guess that "sort out" means cancel and trigger a refund nobody asked for.

> **Skeptic's corner** is this just prompt fiddling dressed up? No. It is interface design. A tool description is an API contract the model consumes at runtime. You would not ship a public function called `handle` with the docstring "handle the booking". Hold tools to the same bar.

---
## 4. The pattern generalises past airlines

Nothing here is airline-specific. A tool is any function you would rather run than let the model guess at. Models are unreliable at exact arithmetic and date math, so those are perfect tools. Here is a general-purpose one, unrelated to bookings, plugged into the same machinery.

```mermaid
flowchart LR
    Q[how long is my 11:20 to 14:05 layover?] --> M[model: minutes_between]
    M --> T[tool computes exactly: 165]
    T --> A[answer grounded in real math]
```

> **What runs next** a `minutes_between` tool that does time math, used by an agent.
> **Python construct** parsing strings to integers, plain arithmetic, returning a typed value.
> **LLM concept** offloading. Give the model a tool for anything it would fumble, and its job shrinks to knowing when to call it.

In [5]:
@tool
def minutes_between(start: str, end: str) -> int:
    '''Minutes between two 24-hour times formatted HH:MM, for example 11:20 and 14:05.'''
    def to_min(t):
        h, m = t.split(":")
        return int(h) * 60 + int(m)
    return to_min(end) - to_min(start)


model = ScriptedChatModel(responses=[
    AIMessage(content="", tool_calls=[{"name": "minutes_between", "args": {"start": "11:20", "end": "14:05"}, "id": "x", "type": "tool_call"}]),
    AIMessage(content="Your layover is 165 minutes, about 2 hours 45 minutes. Enough time to clear the terminal."),
])

agent = create_agent(model, tools=[minutes_between], system_prompt="You are a travel assistant.")
result = agent.invoke({"messages": [{"role": "user", "content": "Layover from 11:20 to 14:05, is that enough?"}]})
show_trace(result)

HumanMessage   -> Layover from 11:20 to 14:05, is that enough?
AIMessage      -> calls minutes_between({'start': '11:20', 'end': '14:05'})
ToolMessage    -> 165
AIMessage      -> Your layover is 165 minutes, about 2 hours 45 minutes. Enough time to clear the terminal.


> **What just happened** the same loop, a totally different domain. The model asked for a calculation instead of doing it, so the 165 is correct by construction, not by luck. Swap in a currency tool, a database query, or a search, and the shape never changes: describe the tool well, let the model decide when to reach for it.

---
## 5. Tools fail, and the model must see the failure

Tools call the real world, and the real world returns errors. A tool that hides its failure produces an agent that confidently lies. The fix is to return the error as the tool result, so the model can react to it.

```mermaid
flowchart TD
    M[model: lookup_pnr ZZZZZZ] --> T[tool: PNR not found]
    T --> M2[model: apologise, ask for a correct PNR]
```

> **What runs next** call the lookup tool with a bad PNR, so the tool returns a not-found message, and let the model recover.
> **Python construct** a graceful return path instead of an unhandled exception.
> **Gotcha** an uncaught exception can crash the whole loop. A returned error string keeps the agent alive and lets it recover in words.

In [6]:
model = ScriptedChatModel(responses=[
    AIMessage(content="", tool_calls=[{"name": "lookup_pnr", "args": {"pnr": "ZZZZZZ"}, "id": "e", "type": "tool_call"}]),
    AIMessage(content="I could not find booking ZZZZZZ. Please recheck the six-character PNR on your ticket."),
])

agent = create_agent(model, tools=[lookup_pnr], system_prompt="You are TravelMind.")
result = agent.invoke({"messages": [{"role": "user", "content": "Check my booking ZZZZZZ"}]})
show_trace(result)

HumanMessage   -> Check my booking ZZZZZZ
AIMessage      -> calls lookup_pnr({'pnr': 'ZZZZZZ'})
ToolMessage    -> PNR not found
AIMessage      -> I could not find booking ZZZZZZ. Please recheck the six-character PNR on your ticket.


> **What just happened** the tool returned "PNR not found" instead of throwing. The model read that and turned it into a helpful correction. The loop stayed alive. This is the difference between an agent that degrades gracefully and one that dies on the first typo.

> **Gotcha** decide up front what a tool returns on failure, and make it something the model can act on. "PNR not found, ask the user to recheck" beats a stack trace every time.

---
## What you can now do

- Read the live schema a tool exposes to the model: name, description, args.
- Run an agent across several tools and trace every claim back to a tool result.
- Write tool descriptions that make selection unambiguous.
- Reuse the exact same machinery for any domain, not just airlines.
- Return tool failures as results so the agent recovers instead of crashing.

**Next, Notebook 4.** We add the things that turn a loop into a product: structured output you can trust in code, memory across turns, and middleware that redacts private data, summarises long histories, and pauses for human approval before a risky action.

> **Skeptic's corner to carry forward** most agent bugs are not model bugs, they are interface bugs: a weak docstring, a swallowed error, a tool that returns mush. Fix the interface before you blame the model.